In [31]:
from futu import *
import pandas_ta as ta
import pandas as pd
import glob

import json
import talib

from collections import OrderedDict
import numpy as np

In [32]:
last_date = datetime.today() - timedelta(days=1)
last_date = last_date.strftime('%Y-%m-%d')

In [33]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', None)

In [34]:
quote_ctx = OpenQuoteContext(host='127.0.0.1', port=11111)

2025-07-04 15:15:51,248 | 36440 | [open_context_base.py] _send_init_connect_sync:311: InitConnect ok: conn_id=7, host=127.0.0.1, port=11111, user_id=11541642


In [35]:
# quote_ctx.close()

In [ ]:
def get_watch_tickers(group_name="特别关注", quote_ctx=None):
	if quote_ctx is None:
		raise ValueError("quote_ctx cannot be None")
	ret_code, heart_watch_list = quote_ctx.get_user_security(group_name=group_name)

	heart_watch_list = heart_watch_list[heart_watch_list['stock_type'] == 'STOCK']
	heart_watch_list = heart_watch_list[heart_watch_list['code'].str.startswith('HK')]

	watch_df = heart_watch_list[['code']]
	watch_df = watch_df.rename(columns={'code': 'ticker_id'})

	return watch_df

In [38]:
trd_ctx = OpenSecTradeContext(filter_trdmarket=TrdMarket.HK, host='127.0.0.1', port=11111, security_firm=SecurityFirm.FUTUSECURITIES)

ret, position_df = trd_ctx.position_list_query()

position_tickers = position_df[['code']]
position_tickers = position_tickers.rename(columns={'code': 'ticker_id'})

2025-07-04 15:15:57,891 | 36440 | [open_context_base.py] _send_init_connect_sync:311: InitConnect ok: conn_id=8, host=127.0.0.1, port=11111, user_id=11541642


In [39]:
display(position_df)

,code,stock_name,position_market,qty,can_sell_qty,cost_price,cost_price_valid,average_cost,diluted_cost,market_val,...,today_buy_qty,today_buy_val,today_pl_val,today_trd_val,today_sell_qty,today_sell_val,position_side,unrealized_pl,realized_pl,currency
0,HK.03690,美团-W,HK,500.0,500.0,132.500,True,132.500,132.500,60350.0,...,0.0,0.0,-1050.0,0.0,0.0,0.0,LONG,-5900.0000,0.0000,HKD
1,HK.01788,国泰君安国际,HK,10000.0,10000.0,4.730,True,4.730,4.730,42800.0,...,0.0,0.0,4200.0,0.0,0.0,0.0,LONG,-4500.0000,0.0000,HKD
2,HK.01398,工商银行,HK,5000.0,5000.0,6.360,True,6.360,6.360,30700.0,...,0.0,0.0,250.0,0.0,0.0,0.0,LONG,-1100.0000,0.0000,HKD
3,HK.01211,比亚迪股份,HK,1000.0,1000.0,265.600,True,265.600,265.600,121300.0,...,0.0,0.0,-1700.0,0.0,0.0,0.0,LONG,-144300.0000,0.0000,HKD
4,HK.00700,腾讯控股,HK,283.0,283.0,62.259,True,267.338,62.259,140594.4,...,0.0,0.0,-1188.6,0.0,0.0,0.0,LONG,64937.6635,556180.7365,HKD


In [40]:
def get_plate_stock(group_name, quote_ctx=None):
    ret, result_df = quote_ctx.get_plate_stock(group_name)  # 恒指成分
    result_df = result_df[['code']]
    result_df = result_df.rename(columns={'code': 'ticker_id'})

    return result_df

hengsheng_df = get_plate_stock('HK.HSI Constituent Stocks', quote_ctx)
hot_df = get_plate_stock('HK.LIST1600', quote_ctx)
bank_df = get_plate_stock("HK.LIST1079", quote_ctx)

biology_df = get_plate_stock("HK.LIST1050", quote_ctx)
medicine_df = get_plate_stock("HK.LIST1067", quote_ctx)
car_df = get_plate_stock("HK.LIST1067", quote_ctx)

In [ ]:
local_files = glob.glob('./data/*.csv')
local_files = [item.replace('.csv', '').replace('./data/', '') for item in local_files]

local_tickers = pd.read_csv('./conf/ticker_id.csv')
local_tickers = pd.concat([local_tickers, pd.DataFrame({'ticker_id': local_files})], ignore_index=True)

watch_df_sep = get_watch_tickers(group_name="特别关注", quote_ctx=quote_ctx)
watch_df_hk = get_watch_tickers(group_name="港股", quote_ctx=quote_ctx)
watch_df_a = get_watch_tickers(group_name="沪深", quote_ctx=quote_ctx)

config_tickers = pd.concat([local_tickers, 
			    hengsheng_df, bank_df, hot_df, biology_df, 
			    medicine_df, car_df, position_tickers,
			    watch_df_sep, watch_df_hk, watch_df_a], ignore_index=True)
config_tickers = config_tickers.drop_duplicates(subset=['ticker_id'])

config_tickers.to_csv('./conf/ticker_id.csv', index=False, encoding='utf-8-sig')

display(config_tickers.shape)

TypeError: string indices must be integers

In [ ]:
def download_ticker_ohcl(ticker_id):
	output_file = f"./data/{ticker_id}.csv"
	max_date = None
	curr_date = datetime.today().date().strftime('%Y-%m-%d')
	last_date = (datetime.today() - timedelta(days=1)).date().strftime('%Y-%m-%d')
	curr_hour = datetime.now().hour
	"""
	if os.path.isfile(output_file):
		data = pd.read_csv(output_file)
		if not data.empty:
			max_date = data['time_key'].max()
			max_date = pd.to_datetime(max_date).strftime('%Y-%m-%d')
			if (curr_hour > 18 and max_date >= curr_date) or (curr_hour <= 18 and max_date >= last_date):  
				# 18点以后，尝试获取今天的数据,18点之前，尝试获取昨天的数据
				return data
	"""

	ret, data, page_req_key = quote_ctx.request_history_kline(
		ticker_id, start='2000-01-01', end='2025-12-31', max_count=10000, session=Session.ALL)
	time.sleep(0.51)   # qps limit 2
	if ret != 0:
		print(f'Error fetching history for {ticker_id}: {data}')
		return None
	
	data.to_csv(output_file, index=False)
	return data

In [43]:
# Initialize an empty DataFrame to store ticker history

df_ohcl = pd.DataFrame({
    key: [] for key in
    ['code', 'name', 'time_key', 'open', 'close', 'high', 'low', 'pe_ratio',
       'turnover_rate', 'volume', 'turnover', 'change_rate', 'last_close']
})

In [44]:
for index, row in config_tickers.iterrows():
    ticker_id = row['ticker_id']
    try:
        curr_history = download_ticker_ohcl(ticker_id)
        if curr_history is not None and not curr_history.empty:
            df_ohcl = pd.concat([df_ohcl, curr_history], axis=0)
            print(f"Processed {ticker_id} {index + 1}/{config_tickers.shape[0]}, {curr_history.shape}, {curr_history['time_key'].max()}")
        else:
            print(f"Processed {ticker_id} {index + 1}/{config_tickers.shape[0]}, ticker_data empty.")
    except Exception as e:
        print(ticker_id, e)

Processed SH.601088 1/732, (2371, 13), 2025-07-04 00:00:00
Processed SH.600585 2/732, (2435, 13), 2025-07-04 00:00:00
Processed SZ.002714 3/732, (2425, 13), 2025-07-04 00:00:00
Processed SH.601899 4/732, (2429, 13), 2025-07-04 00:00:00
Processed SZ.000651 5/732, (2278, 13), 2025-07-04 00:00:00
Processed SZ.000333 6/732, (2385, 13), 2025-07-04 00:00:00
Processed SH.600019 7/732, (2345, 13), 2025-07-04 00:00:00
Processed SH.600010 8/732, (2426, 13), 2025-07-04 00:00:00
Processed SH.600438 9/732, (2360, 13), 2025-07-04 00:00:00
Processed HK.02276 10/732, (869, 13), 2025-07-04 00:00:00
Processed HK.02510 11/732, (164, 13), 2025-07-04 00:00:00
Processed HK.00887 12/732, (2464, 13), 2025-07-04 00:00:00
Processed HK.01541 13/732, (448, 13), 2025-07-04 00:00:00
Processed HK.00688 14/732, (2464, 13), 2025-07-04 00:00:00
Processed HK.09690 15/732, (434, 13), 2025-07-04 00:00:00
Processed HK.01347 16/732, (2464, 13), 2025-07-04 00:00:00
Processed HK.00059 17/732, (2383, 13), 2025-03-31 00:00:00
P

In [59]:
print(df_ohcl['code'].drop_duplicates().shape)

quote_ctx.close()

(730,)


In [46]:
def super_trend_daily(row, df_ohcl, test=None):
	ticker_id = row['ticker_id']
	ticker_data = df_ohcl[df_ohcl['code'] == ticker_id].reset_index(drop=True)
	ticker_data = ticker_data.assign(
		supertd_change = 0,
		supertd_prev = 0,
		supertl=0,
		supertd=0,
		supert=0,
	)

	row['ticker_data'] = None
	if ticker_data is None or ticker_data.empty:
		print(f"Failed to fetch ticker_data for {ticker_id}")
		return row


	SUPERTd = ta.supertrend(high=ticker_data['high'], low=ticker_data['low'], close=ticker_data['close'],
			 length=10, multiplier=3.0, fillna=0.0)

	if SUPERTd is None or SUPERTd.empty:
		print(f"Failed to calculate Supertrend for {ticker_id} | 可能是上市时间太短。 {ticker_data.shape}")
		return row
	
	ticker_data['supert'] = SUPERTd['SUPERT_10_3.0']
	ticker_data['supertl'] = SUPERTd['SUPERTl_10_3.0']
	ticker_data['supertd'] = SUPERTd['SUPERTd_10_3.0']

	ticker_data['supertd_prev'] = ticker_data['supertd'].shift(1)

	ticker_data = ticker_data.assign(
		supertd_change = 0
	)
	condition = (ticker_data['supertd'] > 0) & (ticker_data['supertd_prev'] < 0)
	ticker_data.loc[condition, 'supertd_change'] = 1

	condition = (ticker_data['supertd'] < 0) & (ticker_data['supertd_prev'] > 0)
	ticker_data.loc[condition, 'supertd_change'] = -1

	row['ticker_data'] = ticker_data
	return row

In [47]:
def cal_technicals(row, ticker_data_df, test=None):
	ticker_data = row['ticker_data']
	if ticker_data is None:
		return row

	ticker_data = ticker_data.assign(
		EMA_13_daily=None,
		EMA_21_daily=None,
		EMA_144_daily=None,
		EMA_169_daily=None,
		SMA_5_daily=None,
		SMA_10_daily=None,
		SMA_20_daily=None,
		SMA_50_daily=None,
		SMA_100_daily=None,
		vegas=0,
	)
	row['ticker_data'] = None
	if ticker_data.empty:
		return row
	
	indicators = [
		(talib.EMA, "EMA_13_daily", 13),
		(talib.EMA, "EMA_21_daily", 20),
		(talib.EMA, "EMA_144_daily", 144),
		(talib.EMA, "EMA_169_daily", 169),
		(talib.SMA, "SMA_5_daily", 5),
		(talib.SMA, "SMA_10_daily", 10),
		(talib.SMA, "SMA_20_daily", 20),
		(talib.SMA, "SMA_50_daily", 50),
		(talib.SMA, "SMA_100_daily", 100),
	]

	for func, name, period in indicators:
		ticker_data[name] = func(ticker_data['close'], timeperiod=period)

	ticker_data = ticker_data.assign(SMA_5_Cross_SMA_20=0,)
	ticker_data['SMA_5_daily_prev'] = ticker_data['SMA_5_daily'].shift(1)
	ticker_data['SMA_20_daily_prev'] = ticker_data['SMA_20_daily'].shift(1)

	condition = (ticker_data['close'] > ticker_data['EMA_144_daily']) & (ticker_data['close'] > ticker_data['EMA_169_daily'])
	ticker_data.loc[condition, 'vegas'] = 1

	condition = (ticker_data['close'] < ticker_data['EMA_144_daily']) & (ticker_data['close'] < ticker_data['EMA_169_daily'])
	ticker_data.loc[condition, 'vegas'] = -1

	condition = (ticker_data['SMA_5_daily'] > ticker_data['SMA_20_daily']) & (ticker_data['SMA_5_daily_prev'] < ticker_data['SMA_20_daily_prev'])
	ticker_data.loc[condition, 'SMA_5_Cross_SMA_20'] = 1
	condition = (ticker_data['SMA_5_daily'] < ticker_data['SMA_20_daily']) & (ticker_data['SMA_5_daily_prev'] > ticker_data['SMA_20_daily_prev'])
	ticker_data.loc[condition, 'SMA_5_Cross_SMA_20'] = -1

	ticker_data = ticker_data.drop(columns=['SMA_5_daily_prev', 'SMA_20_daily_prev'])

	row['ticker_data'] = ticker_data
	return row

In [62]:
df = config_tickers.apply(super_trend_daily, axis=1, args=(df_ohcl, None))

print('cal super trend success.')

cal super trend success.


In [49]:
df = df.apply(cal_technicals, axis=1, args=(df_ohcl, None))
print('cal technicals success.')

cal technicals success.


In [50]:
ticker_result_df = pd.DataFrame()

for index, row in df.iterrows():
    if row['ticker_data'] is None or row['ticker_data'].empty:
        continue
    ticker_result_df = pd.concat([ticker_result_df, row['ticker_data']], axis=0)

In [51]:
ticker_result_df = ticker_result_df.dropna(subset=['EMA_169_daily'])
ticker_result_df = ticker_result_df.sort_values(by=['code', 'time_key']).reset_index(drop=True)

In [52]:
ticker_result_df = ticker_result_df[ticker_result_df['time_key'] >= last_date]  # 只保留最近的数据
ticker_result_df = ticker_result_df.sort_values(by=['code', 'time_key']).reset_index(drop=True)

In [53]:
technical_df = ticker_result_df[ticker_result_df['SMA_5_Cross_SMA_20'].rolling(10).sum() > 0]
technical_df = technical_df.reset_index(drop=True)
display(technical_df.sort_values(by='name').reset_index(drop=True))

,code,name,time_key,open,close,high,low,pe_ratio,turnover_rate,volume,...,EMA_21_daily,EMA_144_daily,EMA_169_daily,SMA_5_daily,SMA_10_daily,SMA_20_daily,SMA_50_daily,SMA_100_daily,vegas,SMA_5_Cross_SMA_20
0,HK.02505,EDA集团控股,2025-07-04 00:00:00,2.67,2.61,2.70,2.61,22.695,0.00032,141000.0,...,2.650221,2.830227,2.898866,2.620,2.615,2.64250,2.7145,2.76035,-1,0
1,HK.01070,TCL电子,2025-07-03 00:00:00,9.70,10.02,10.06,9.65,14.355,0.00312,7858171.0,...,9.632721,8.250502,7.969807,9.578,9.466,9.54500,9.8690,8.78030,1,1
2,HK.01070,TCL电子,2025-07-04 00:00:00,10.02,9.64,10.06,9.57,13.810,0.00313,7894081.0,...,9.633414,8.269667,7.989456,9.656,9.510,9.54450,9.8946,8.80960,1,0
3,HK.02522,一脉阳光,2025-07-04 00:00:00,13.96,13.76,14.06,13.52,0.000,0.02917,7533000.0,...,14.375244,24.502482,25.284784,13.664,13.556,13.92400,17.1680,25.10820,-1,0
4,HK.02522,一脉阳光,2025-07-03 00:00:00,13.60,13.90,14.36,13.48,0.000,0.05139,13272000.0,...,14.440006,24.652726,25.421984,13.580,13.470,14.07800,17.3288,25.30560,-1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
228,HK.08629,集信国控,2025-07-03 00:00:00,9.20,9.48,9.48,9.18,18.265,0.00172,17500.0,...,9.485184,9.595523,9.610985,9.178,9.345,9.60000,9.5402,9.57380,-1,0
229,HK.08629,集信国控,2025-07-04 00:00:00,9.20,9.30,9.32,9.13,17.919,0.00167,17000.0,...,9.467547,9.591447,9.607326,9.180,9.326,9.58650,9.5438,9.56360,-1,0
230,HK.01942,马可数字科技,2025-07-03 00:00:00,0.59,0.59,0.60,0.57,0.000,0.00620,7092000.0,...,0.547604,0.792934,1.010357,0.616,0.586,0.54325,0.4338,0.54990,-1,0
231,HK.01942,马可数字科技,2025-07-04 00:00:00,0.59,0.57,0.59,0.56,0.000,0.00635,7266000.0,...,0.549737,0.789859,1.005177,0.612,0.590,0.54925,0.4380,0.54590,-1,0


In [54]:
bull_df = ticker_result_df[(ticker_result_df['supertd'] > 0) & (ticker_result_df['supertd_change'] > 0) & (ticker_result_df['vegas'] > 0)]
bull_df = bull_df.sort_values(by=['code', 'time_key']).reset_index(drop=True)

In [55]:
supertd_vol_big = bull_df[bull_df['volume'] >= 15000000]  # 非常小体量的不考虑
supertd_vol_big = supertd_vol_big.reset_index(drop=True)
display(supertd_vol_big)

,code,name,time_key,open,close,high,low,pe_ratio,turnover_rate,volume,...,EMA_21_daily,EMA_144_daily,EMA_169_daily,SMA_5_daily,SMA_10_daily,SMA_20_daily,SMA_50_daily,SMA_100_daily,vegas,SMA_5_Cross_SMA_20
0,HK.00697,首程控股,2025-07-03 00:00:00,1.64,1.70,1.74,1.63,30.357,0.01845,134376600.0,...,1.608394,1.449615,1.427409,1.626,1.604,1.6095,1.6034,1.551007,1,0
1,HK.01108,凯盛新能,2025-07-04 00:00:00,4.95,5.11,5.27,4.83,0.000,0.14873,37182000.0,...,4.291457,3.936209,3.962584,4.606,4.448,4.2105,3.8990,3.809300,1,0
2,HK.02333,长城汽车,2025-07-04 00:00:00,12.80,13.08,13.40,12.70,8.169,0.01499,34760466.0,...,12.320613,12.549832,12.522327,12.544,12.284,12.2570,12.1376,12.702000,1,0


In [56]:
supertd_vol_mini = bull_df[(bull_df['volume'] < 15000000) & (bull_df['change_rate'] > 0.01)]
supertd_vol_mini = supertd_vol_mini.reset_index(drop=True)

# 小体量风险大，同时考虑交易量，确认稳定性
display(supertd_vol_mini)

,code,name,time_key,open,close,high,low,pe_ratio,turnover_rate,volume,...,EMA_21_daily,EMA_144_daily,EMA_169_daily,SMA_5_daily,SMA_10_daily,SMA_20_daily,SMA_50_daily,SMA_100_daily,vegas,SMA_5_Cross_SMA_20
0,HK.08606,倢冠控股,2025-07-03 00:00:00,0.138,0.128,0.138,0.138,0.0,0.00003,40000.0,...,0.103806,0.112107,0.110606,0.104,0.0995,0.1031,0.11172,0.12879,1,1


In [57]:
bear_df = ticker_result_df[(ticker_result_df['supertd'] < 0) & (ticker_result_df['supertd_change'] < 0) & (ticker_result_df['vegas'] < 0)]
bear_df = bear_df.sort_values(by=['code', 'time_key']).reset_index(drop=True)
# display(bear_df)

In [58]:
supertd_tech_df = ticker_result_df[
	(ticker_result_df['supertd'] > 0) & (ticker_result_df['supertd_change'] > 0) & (ticker_result_df['vegas'] > 0)
	& (ticker_result_df['SMA_5_Cross_SMA_20'].rolling(window=10).sum() > 0)]

supertd_tech_df = supertd_tech_df.sort_values(by=['code', 'time_key']).reset_index(drop=True)
# 符合supertd条件，且近期SMA有上穿
supertd_tech_df[supertd_tech_df['time_key'] >= last_date].sort_values(by='name').reset_index(drop=True)
bull_tech_vol_df = supertd_tech_df[supertd_tech_df['volume'] >= 15000000]  # 非常小体量的不考虑

display(bull_tech_vol_df)

,code,name,time_key,open,close,high,low,pe_ratio,turnover_rate,volume,...,EMA_21_daily,EMA_144_daily,EMA_169_daily,SMA_5_daily,SMA_10_daily,SMA_20_daily,SMA_50_daily,SMA_100_daily,vegas,SMA_5_Cross_SMA_20
